In [23]:
import pandas as pd
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

In [43]:
# Creating mock data for testing

np.random.seed(42)
n_users = 5000
n_days = 30

users = np.arange(1, n_users + 1)
dates = pd.date_range("2024-04-01", periods=n_days, freq="D")

user_df = pd.DataFrame({
    "user_id": users,
    "region": np.random.choice(["North", "South", "East", "West"], n_users),
    "device": np.random.choice(["mobile", "desktop"], n_users, p=[0.7, 0.3]),
    "browser": np.random.choice(
        ["Chrome", "Safari", "Firefox", "Edge"],
        n_users,
        p=[0.6, 0.2, 0.15, 0.05]
    )
})

events = []

event_types = ["page_view", "click", "add_to_cart", "purchase"]

for user in users:
    for date in dates:
        n_events = np.random.poisson(3)

        for _ in range(n_events):
            event = np.random.choice(
                event_types,
                p=[0.6, 0.25, 0.1, 0.05]
            )

            events.append([
                user,
                date,
                date + pd.to_timedelta(np.random.randint(0, 86400), unit="s"),
                event
            ])

df = pd.DataFrame(
    events,
    columns=["user_id", "date", "timestamp", "event_type"]
)

df = df.merge(user_df, on="user_id", how="left")

user_purchase = df.groupby("user_id")["event_type"].apply(
    lambda x: (x == "purchase").any()
).reset_index()

user_purchase.columns = ["user_id", "purchased"]
user_purchase["purchased"] = user_purchase["purchased"].astype(int)

df = df.merge(user_purchase, on="user_id", how="left")

print(df.head())
print(df.info())

   user_id       date           timestamp event_type region   device browser  \
0        1 2024-04-01 2024-04-01 18:35:09  page_view   East  desktop  Chrome   
1        1 2024-04-02 2024-04-02 05:11:20   purchase   East  desktop  Chrome   
2        1 2024-04-02 2024-04-02 08:25:46  page_view   East  desktop  Chrome   
3        1 2024-04-02 2024-04-02 06:30:12  page_view   East  desktop  Chrome   
4        1 2024-04-02 2024-04-02 08:33:24  page_view   East  desktop  Chrome   

   purchased  
0          1  
1          1  
2          1  
3          1  
4          1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449939 entries, 0 to 449938
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     449939 non-null  int64         
 1   date        449939 non-null  datetime64[ns]
 2   timestamp   449939 non-null  datetime64[ns]
 3   event_type  449939 non-null  object        
 4   region      449939 n

In [44]:
effect_size = sms.proportion_effectsize(0.12, 0.15)   

required_n = sms.NormalIndPower().solve_power(
    effect_size, 
    power=0.8, 
    alpha=0.05, 
    ratio=1
    )                                                  

required_n = ceil(required_n)                                                    

print(required_n)

2031


In [51]:
user_df["group"] = np.random.choice(
    ["Treatment", "Control"],
    size=len(user_df),
    p=[0.5, 0.5]
)

In [54]:
exp_users = df.merge(user_df[["user_id", "group"]], on="user_id", how="left")